# تجربة WeMM-Embedding بنفسك

دفتر للعب والفهم. **شغّل الخلايا بالترتيب.** الخلية الأولى تحمّل الموديل
(~17 ثانية و 18GB ذاكرة) وتبقيه محمّلًا، فبقية الخلايا تشتغل فورًا مهما عدّلت وأعدت.

> نصيحة: بعد ما تفهم كل قسم، **غيّر النصوص فيه وأعد التشغيل**. الفهم يجي من كسر الأشياء.

## 0 — التحميل (مرة وحدة)

In [ ]:
import sys, os, time, warnings, logging
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore'); logging.disable(logging.WARNING)

import numpy as np
from src.providers.wemm_embed import embed, load_model

t0 = time.time()
model = load_model()          # tencent/WeMM-Embedding-9B
print(f'جاهز في {time.time()-t0:.1f}s على الجهاز: {model.device}')

## 1 — ما هو الـ embedding أصلاً؟

جملة واحدة تدخل، ويطلع منها **متجه من 4096 رقمًا**. هذا كل شي.

In [ ]:
v = embed(['القهوة العربية تُقدَّم مع التمر.'])[0]

print('الطول:', len(v))
print('أول 8 أرقام:', np.round(v[:8], 4))
print('معيار L2 :', round(float(np.linalg.norm(v)), 4), '  ← مطبّع، فالضرب النقطي = cosine مباشرة')

## 2 — التشابه: الرقم اللي يهم

بما إن المتجهات مطبّعة، التشابه = ضرب نقطي بسيط. القيمة بين ‎-1 و 1.

**عدّل الجمل تحت وشوف كيف تتغير الأرقام.**

In [ ]:
def sim(a, b, **kw):
    """تشابه cosine بين نصين."""
    x, y = embed([a, b], **kw)
    return float(np.dot(x, y))


أزواج = [
    ('القطة تجلس على السجادة.',  'الهرة مستلقية على البساط.'),   # نفس المعنى، كلمات مختلفة
    ('القطة تجلس على السجادة.',  'A cat is sitting on the rug.'), # نفس المعنى، لغتان
    ('القطة تجلس على السجادة.',  'معدل التضخم ارتفع هذا الربع.'), # لا علاقة
]

for a, b in أزواج:
    print(f'{sim(a, b):.4f}   {a[:32]:<34} | {b[:38]}')

## 3 — تحقّق من افتراضاتك ⚠️

أكثر الموديلات الحديثة **غير متماثلة**: تمرّر الاستعلام بقالب مختلف عن المستند،
لأن «كيف أطبخ الكبسة؟» و«الكبسة تُطبخ كذا» يلعبان دورين مختلفين. `sentence-transformers`
يوفّر `encode_query` و `encode_document` لهذا الغرض.

**لكن لا تفترض — تحقّق.** الخلية تحت تسأل الموديل نفسه.

In [ ]:
print('القوالب المسجّلة (prompts):', model.prompts)
print('القالب الافتراضي        :', model.default_prompt_name)

سؤال  = 'كيف أستخرج البيانات من فاتورة ممسوحة ضوئيًا؟'
مستند = 'OCR pipelines convert scanned invoices into structured JSON data.'

q_query = embed([سؤال], mode='query')[0]
q_doc   = embed([سؤال], mode='document')[0]
d       = embed([مستند], mode='document')[0]

فرق = float(np.abs(np.array(q_query) - np.array(q_doc)).max())
print(f'\nأقصى فرق بين متجهَي الوضعين: {فرق:.2e}')
print(f'query   → document : {np.dot(q_query, d):.4f}')
print(f'document→ document : {np.dot(q_doc,   d):.4f}')

if فرق < 1e-6:
    print('\n🔍 النتيجة: prompts فاضية، فالوضعان متطابقان — هذا الموديل **متماثل**.')
    print('   لو بنيت نظام استرجاع بافتراض العكس، بتضيع وقتك على تحسين ما له أثر.')

> **الدرس:** بطاقة الموديل تعرض `encode_query`/`encode_document` في مثالها،
> وهذا يوحي بعدم التماثل. لكن `prompts: {}` في `config_sentence_transformers.json`
> يعني إن الوضعين يمرّان بنفس القالب. **سطر تحقق واحد وفّر عليك افتراضًا خاطئًا.**
>
> أبقينا `mode` في المزود لأنه الواجهة الرسمية — لو سجّل المزوّد قوالب لاحقًا، يشتغل تلقائيًا.

## 4 — البحث الحقيقي: رتّب مستندات حسب سؤال

**هنا العب أكثر شي.** غيّر `سؤالي` وزد مستندات على القائمة وأعد التشغيل.

In [ ]:
سؤالي = 'كيف أستخرج البيانات من فاتورة ممسوحة ضوئيًا؟'

مستنداتي = [
    'OCR pipelines convert scanned invoices into structured JSON data.',
    'نماذج اللغة الكبيرة تُستخدم في استخراج النصوص من المستندات الممسوحة ضوئيًا.',
    'وصفة الكبسة بالدجاج مع البهارات والأرز البسمتي.',
    'The Eiffel Tower is located in Paris, France.',
]

q = np.array(embed([سؤالي], mode='query')[0])
D = np.array(embed(مستنداتي, mode='document'))
نتائج = sorted(enumerate(D @ q), key=lambda x: -x[1])

print(f'❓ {سؤالي}\n' + '─'*72)
for المرتبة, (i, درجة) in enumerate(نتائج, 1):
    print(f'{المرتبة}. {درجة:.4f}  {مستنداتي[i][:60]}')

## 5 — Matryoshka: اقطع المتجه ووفّر ذاكرة

المتجه مدرَّب بحيث **أول N بُعد لحالها تمثيل صالح**. جرّب كم تقدر تقصّ
قبل ما يتكسّر الترتيب.

In [ ]:
مرجع = np.argmax(D @ q)     # الأفضل بالأبعاد الكاملة

for بُعد in (64, 128, 256, 512, 1024, 2048, 4096):
    qd = np.array(embed([سؤالي],  mode='query',    truncate_dim=بُعد)[0])
    Dd = np.array(embed(مستنداتي, mode='document', truncate_dim=بُعد))
    s  = Dd @ qd
    حالة = '✅' if np.argmax(s) == مرجع else '❌ انكسر'
    print(f'{بُعد:>5}d  ذاكرة/متجه: {بُعد*4/1024:>6.1f} KB  أعلى={s.max():.4f}  {حالة}')

## 6 — المتعدد الوسائط: صورة ونص في نفس الفضاء 🖼️

هذي قدرة WeMM اللي ما يقدر عليها أي embedding نصي. تمرّر الصورة كـ `dict`
بدل نص — رابط أو مسار محلي — وتقارنها بنص عربي مباشرة.

> يحتاج إنترنت لتحميل صورة المثال. حط `مسار محلي` بدلها لو تبي.

In [ ]:
from src.providers.wemm_embed import load_model
m = load_model()

صورة = {'image': '/Users/meshalaldalbahi/Downloads/WeMM-Embedding/dataset/image for test embedding .png'}

أوصاف = [
    'صورة عبارة عن واجهة مبني كواجهة اعلانية',
    'صحن كبسة بالدجاج على طاولة.',
    'قطة تنام على أريكة.',
]

v_صورة = m.encode_document([صورة])[0]
v_نص   = m.encode_query(أوصاف)

print('أي وصف عربي يطابق الصورة؟\n' + '─'*54)
for وصف, درجة in sorted(zip(أوصاف, v_نص @ v_صورة), key=lambda x: -x[1]):
    print(f'  {درجة:.4f}  {وصف}')